# Data Engineering et Machine Learning avec Snowflake

## Workshop : Prediction des prix immobiliers

Pipeline ML complet :
1. Configuration de l'environnement (Architecture Medallion)
2. Ingestion des donnees (Bronze)
3. Transformation et typage (Silver)
4. Encodage et preparation (Gold)
5. Exploration des donnees (EDA)
6. Preparation des features ML
7. Entrainement des modeles
8. Evaluation et comparaison
9. Optimisation (Grid Search)
10. Enregistrement dans le Model Registry
11. Inference sur nouvelles donnees

---
## 0. Installation et imports

Installation des dependances et import de toutes les bibliotheques utilisees dans ce notebook.

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'snowflake-ml-python', '--quiet'], check=True)
print("Installation terminee.")

In [ ]:
# Bibliotheques standard
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Snowpark
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col, avg, min as min_, max as max_,
    count, when, lit
)

# Scikit-learn — preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline

# Scikit-learn — modeles
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Scikit-learn — metriques regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Scikit-learn — metriques classification
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)

# Snowflake ML
from snowflake.ml.registry import Registry
from snowflake.ml.model.type_hints import TargetPlatform

print("Toutes les bibliotheques importees avec succes.")

---
## 1. Configuration de l'environnement

Architecture **Medallion** :
- **Bronze** : donnees brutes JSON telles quelles depuis S3
- **Silver** : donnees typees et nettoyees
- **Gold** : donnees encodees et pretes pour le ML
- **ML** : modeles, predictions, registry

In [ ]:
%%sql -r env_setup
CREATE DATABASE IF NOT EXISTS HOUSE_PRICE_DB;

CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE_DB.BRONZE;
CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE_DB.SILVER;
CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE_DB.GOLD;
CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE_DB.ML;

CREATE WAREHOUSE IF NOT EXISTS HOUSE_PRICE_WH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND   = 60
    AUTO_RESUME    = TRUE
    COMMENT        = 'Warehouse pour le workshop House Price ML';

USE DATABASE  HOUSE_PRICE_DB;
USE WAREHOUSE HOUSE_PRICE_WH;

SHOW SCHEMAS IN DATABASE HOUSE_PRICE_DB;

In [ ]:
session = get_active_session()
session.use_database('HOUSE_PRICE_DB')
session.use_warehouse('HOUSE_PRICE_WH')
print(f"Database  : {session.get_current_database()}")
print(f"Warehouse : {session.get_current_warehouse()}")

---
## 2. Ingestion des donnees — Couche Bronze

Chargement des donnees brutes depuis S3 **sans aucune transformation**.
Le Bronze conserve les donnees dans leur etat d'origine (JSON VARIANT) pour garantir la tracabilite.

In [ ]:
%%sql -r stage_list
USE SCHEMA HOUSE_PRICE_DB.BRONZE;

CREATE OR REPLACE STAGE HOUSE_PRICE_STAGE
    URL = 's3://logbrain-datalake/datasets/house_price/';

LIST @HOUSE_PRICE_STAGE;

In [ ]:
%%sql -r raw_preview
SELECT $1, $2, $3, $4, $5, $6, $7, $8, $9, $10, $11, $12, $13
FROM @HOUSE_PRICE_DB.BRONZE.HOUSE_PRICE_STAGE
LIMIT 10;

In [ ]:
%%sql -r ff_check
USE SCHEMA HOUSE_PRICE_DB.BRONZE;

CREATE OR REPLACE FILE FORMAT JSON_FORMAT
    TYPE              = 'JSON'
    STRIP_OUTER_ARRAY = TRUE
    STRIP_NULL_VALUES = FALSE
    TRIM_SPACE        = TRUE;

SELECT $1
FROM @HOUSE_PRICE_STAGE
(FILE_FORMAT => 'JSON_FORMAT')
LIMIT 3;

In [ ]:
%%sql -r bronze_load
USE SCHEMA HOUSE_PRICE_DB.BRONZE;

CREATE OR REPLACE TABLE HOUSE_PRICES_RAW (
    RAW_JSON   VARIANT,
    _LOADED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

COPY INTO HOUSE_PRICES_RAW (RAW_JSON)
FROM (
    SELECT $1
    FROM @HOUSE_PRICE_STAGE
)
FILE_FORMAT = JSON_FORMAT;

SELECT COUNT(*) AS nb_lignes_bronze FROM HOUSE_PRICES_RAW;

In [ ]:
%%sql -r bronze_preview
SELECT * FROM HOUSE_PRICE_DB.BRONZE.HOUSE_PRICES_RAW LIMIT 10;

In [ ]:
%%sql -r audit_nulls
SELECT
    COUNT(*) AS total_lignes,

    SUM(CASE WHEN RAW_JSON:"price"::FLOAT             IS NULL THEN 1 ELSE 0 END) AS nulls_prix,
    SUM(CASE WHEN RAW_JSON:"area"::FLOAT              IS NULL THEN 1 ELSE 0 END) AS nulls_surface,
    SUM(CASE WHEN RAW_JSON:"bedrooms"::INT            IS NULL THEN 1 ELSE 0 END) AS nulls_chambres,
    SUM(CASE WHEN RAW_JSON:"bathrooms"::INT           IS NULL THEN 1 ELSE 0 END) AS nulls_salles_de_bain,
    SUM(CASE WHEN RAW_JSON:"stories"::INT             IS NULL THEN 1 ELSE 0 END) AS nulls_etages,
    SUM(CASE WHEN RAW_JSON:"parking"::INT             IS NULL THEN 1 ELSE 0 END) AS nulls_parking,
    SUM(CASE WHEN RAW_JSON:"mainroad"::VARCHAR        IS NULL THEN 1 ELSE 0 END) AS nulls_route_principale,
    SUM(CASE WHEN RAW_JSON:"guestroom"::VARCHAR       IS NULL THEN 1 ELSE 0 END) AS nulls_chambre_amis,
    SUM(CASE WHEN RAW_JSON:"basement"::VARCHAR        IS NULL THEN 1 ELSE 0 END) AS nulls_sous_sol,
    SUM(CASE WHEN RAW_JSON:"hotwaterheating"::VARCHAR IS NULL THEN 1 ELSE 0 END) AS nulls_chauffage,
    SUM(CASE WHEN RAW_JSON:"airconditioning"::VARCHAR IS NULL THEN 1 ELSE 0 END) AS nulls_climatisation,
    SUM(CASE WHEN RAW_JSON:"prefarea"::VARCHAR        IS NULL THEN 1 ELSE 0 END) AS nulls_zone_privilegiee,
    SUM(CASE WHEN RAW_JSON:"furnishingstatus"::VARCHAR IS NULL THEN 1 ELSE 0 END) AS nulls_ameublement,

    ROUND(SUM(CASE WHEN RAW_JSON:"price"::FLOAT IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_nulls_prix,
    ROUND(SUM(CASE WHEN RAW_JSON:"area"::FLOAT  IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_nulls_surface

FROM HOUSE_PRICE_DB.BRONZE.HOUSE_PRICES_RAW;

In [ ]:
%%sql -r medianes
SELECT
    MEDIAN(RAW_JSON:"price"::FLOAT)   AS mediane_prix,
    MEDIAN(RAW_JSON:"area"::FLOAT)    AS mediane_surface,
    MEDIAN(RAW_JSON:"bedrooms"::INT)  AS mediane_chambres,
    MEDIAN(RAW_JSON:"bathrooms"::INT) AS mediane_salles_de_bain,
    MEDIAN(RAW_JSON:"stories"::INT)   AS mediane_etages,
    MEDIAN(RAW_JSON:"parking"::INT)   AS mediane_parking
FROM HOUSE_PRICE_DB.BRONZE.HOUSE_PRICES_RAW
WHERE RAW_JSON:"price"::FLOAT IS NOT NULL;

---
## 3. Typage des donnees — Couche Silver

Dans la couche Silver, on effectue uniquement :
- Le **renommage** des colonnes en francais
- Le **typage** correct de chaque colonne (FLOAT, INT, VARCHAR)
- La **suppression uniquement** des lignes ou `PRIX` est null (variable cible indispensable)
- Pour les autres colonnes : **imputation par la mediane** via COALESCE si des nulls sont detectes
- Ajout des colonnes de **tracabilite**

> Aucun encodage ici — cela sera fait dans la couche Gold.

In [ ]:
%%sql -r silver_create
USE SCHEMA HOUSE_PRICE_DB.SILVER;

CREATE OR REPLACE TABLE HOUSE_PRICES_TYPED AS
SELECT
    -- Typage des colonnes numeriques avec noms francais
    RAW_JSON:"price"::FLOAT                                         AS PRIX,
    COALESCE(RAW_JSON:"area"::FLOAT,
             MEDIAN(RAW_JSON:"area"::FLOAT) OVER ())                AS SURFACE,
    COALESCE(RAW_JSON:"bedrooms"::INT,
             MEDIAN(RAW_JSON:"bedrooms"::INT) OVER ())              AS CHAMBRES,
    COALESCE(RAW_JSON:"bathrooms"::INT,
             MEDIAN(RAW_JSON:"bathrooms"::INT) OVER ())             AS SALLES_DE_BAIN,
    COALESCE(RAW_JSON:"stories"::INT,
             MEDIAN(RAW_JSON:"stories"::INT) OVER ())               AS ETAGES,
    COALESCE(RAW_JSON:"parking"::INT,
             MEDIAN(RAW_JSON:"parking"::INT) OVER ())               AS PARKING,

    -- Colonnes categoriques conservees en VARCHAR
    RAW_JSON:"mainroad"::VARCHAR                                    AS ROUTE_PRINCIPALE,
    RAW_JSON:"guestroom"::VARCHAR                                   AS CHAMBRE_AMIS,
    RAW_JSON:"basement"::VARCHAR                                    AS SOUS_SOL,
    RAW_JSON:"hotwaterheating"::VARCHAR                             AS CHAUFFAGE_EAU_CHAUDE,
    RAW_JSON:"airconditioning"::VARCHAR                             AS CLIMATISATION,
    RAW_JSON:"prefarea"::VARCHAR                                    AS ZONE_PRIVILEGIEE,
    RAW_JSON:"furnishingstatus"::VARCHAR                            AS STATUT_AMEUBLEMENT,

    -- Tracabilite
    _LOADED_AT          AS _BRONZE_LOADED_AT,
    CURRENT_TIMESTAMP() AS _SILVER_LOADED_AT

FROM HOUSE_PRICE_DB.BRONZE.HOUSE_PRICES_RAW
WHERE RAW_JSON:"price"::FLOAT IS NOT NULL;

SELECT COUNT(*) AS nb_lignes_silver FROM HOUSE_PRICES_TYPED;

In [ ]:
%%sql -r silver_check
SELECT
    (SELECT COUNT(*) FROM HOUSE_PRICE_DB.BRONZE.HOUSE_PRICES_RAW)    AS nb_lignes_bronze,
    (SELECT COUNT(*) FROM HOUSE_PRICE_DB.SILVER.HOUSE_PRICES_TYPED)  AS nb_lignes_silver,
    nb_lignes_bronze - nb_lignes_silver                               AS lignes_supprimees,
    ROUND((nb_lignes_bronze - nb_lignes_silver) * 100.0
          / nb_lignes_bronze, 2)                                      AS pct_perte;

In [ ]:
%%sql -r silver_stats
SELECT
    COUNT(*)                  AS nb_lignes,
    COUNT(*) - COUNT(PRIX)    AS nulls_prix,
    COUNT(*) - COUNT(SURFACE) AS nulls_surface,
    MIN(PRIX)                 AS prix_min,
    MAX(PRIX)                 AS prix_max,
    ROUND(AVG(PRIX), 0)       AS prix_moyen,
    MIN(SURFACE)              AS surface_min,
    MAX(SURFACE)              AS surface_max
FROM HOUSE_PRICE_DB.SILVER.HOUSE_PRICES_TYPED;

---
## 4. Encodage et preparation — Couche Gold

Dans la couche Gold, on effectue :
- **Encodage binaire** : yes/no -> 1/0
- **Encodage ordinal** : furnished=2, semi-furnished=1, unfurnished=0

La normalisation (StandardScaler) sera appliquee dans le Pipeline ML a l'etape 9.

In [ ]:
%%sql -r gold_create
USE SCHEMA HOUSE_PRICE_DB.GOLD;

CREATE OR REPLACE TABLE HOUSE_PRICES AS
SELECT
    PRIX,
    SURFACE,
    CHAMBRES,
    SALLES_DE_BAIN,
    ETAGES,
    PARKING,

    -- Encodage binaire : yes -> 1 / no -> 0
    CASE WHEN ROUTE_PRINCIPALE     = 'yes' THEN 1 ELSE 0 END AS ROUTE_PRINCIPALE,
    CASE WHEN CHAMBRE_AMIS         = 'yes' THEN 1 ELSE 0 END AS CHAMBRE_AMIS,
    CASE WHEN SOUS_SOL             = 'yes' THEN 1 ELSE 0 END AS SOUS_SOL,
    CASE WHEN CHAUFFAGE_EAU_CHAUDE = 'yes' THEN 1 ELSE 0 END AS CHAUFFAGE_EAU_CHAUDE,
    CASE WHEN CLIMATISATION        = 'yes' THEN 1 ELSE 0 END AS CLIMATISATION,
    CASE WHEN ZONE_PRIVILEGIEE     = 'yes' THEN 1 ELSE 0 END AS ZONE_PRIVILEGIEE,

    -- Encodage ordinal : furnished=2 / semi-furnished=1 / unfurnished=0
    CASE STATUT_AMEUBLEMENT
        WHEN 'furnished'      THEN 2
        WHEN 'semi-furnished' THEN 1
        ELSE 0
    END AS STATUT_AMEUBLEMENT_ENC

FROM HOUSE_PRICE_DB.SILVER.HOUSE_PRICES_TYPED;

SELECT COUNT(*) AS nb_lignes_gold FROM HOUSE_PRICES;

In [ ]:
%%sql -r gold_preview
SELECT * FROM HOUSE_PRICE_DB.GOLD.HOUSE_PRICES LIMIT 10;

---
## 5. Exploration des donnees (EDA)

Comprehension du dataset avec **Snowpark** puis visualisations Python.

In [ ]:
snow_df = session.table('HOUSE_PRICE_DB.GOLD.HOUSE_PRICES')

print(f"Nombre de lignes : {snow_df.count()}")
print("\nApercu des donnees :")
snow_df.show(10)

In [ ]:
print("Statistiques descriptives :")
snow_df.describe().show()

snow_df.select(
    count(col('PRIX')).alias('nb_lignes'),
    avg(col('PRIX')).alias('prix_moyen'),
    min_(col('PRIX')).alias('prix_min'),
    max_(col('PRIX')).alias('prix_max'),
    avg(col('SURFACE')).alias('surface_moyenne'),
    min_(col('SURFACE')).alias('surface_min'),
    max_(col('SURFACE')).alias('surface_max')
).show()

In [ ]:
print("Prix moyen par statut d'ameublement :")
snow_df.group_by('STATUT_AMEUBLEMENT_ENC') \
       .agg(
           avg(col('PRIX')).alias('PRIX_MOYEN'),
           count(col('PRIX')).alias('NB_MAISONS')
       ) \
       .sort('STATUT_AMEUBLEMENT_ENC') \
       .show()

print("Prix moyen par nombre de chambres :")
snow_df.group_by('CHAMBRES') \
       .agg(
           avg(col('PRIX')).alias('PRIX_MOYEN'),
           count(col('PRIX')).alias('NB_MAISONS')
       ) \
       .sort('CHAMBRES') \
       .show()

In [ ]:
print("Maisons avec climatisation et zone privilegiee (top 10 par prix) :")
snow_df.filter(
    (col('CLIMATISATION') == 1) & (col('ZONE_PRIVILEGIEE') == 1)
).select(
    'PRIX', 'SURFACE', 'CHAMBRES', 'CLIMATISATION', 'ZONE_PRIVILEGIEE'
).sort(col('PRIX').desc()).show(10)

In [ ]:
df = snow_df.to_pandas()
print(f"Shape : {df.shape}")
print(f"\nValeurs manquantes :\n{df.isnull().sum()}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribution des variables numeriques', fontsize=14)
num_cols = ['PRIX', 'SURFACE', 'CHAMBRES', 'SALLES_DE_BAIN', 'ETAGES', 'PARKING']
for i, col_name in enumerate(num_cols):
    ax = axes[i // 3, i % 3]
    ax.hist(df[col_name], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_title(col_name)
    ax.set_xlabel(col_name)
    ax.set_ylabel('Frequence')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    df['SURFACE'], df['PRIX'],
    c=df['CHAMBRES'], cmap='viridis',
    alpha=0.6, edgecolor='black', s=40
)
fig.colorbar(scatter, ax=ax, label='Nombre de chambres')
z = np.polyfit(df['SURFACE'], df['PRIX'], 1)
p = np.poly1d(z)
ax.plot(sorted(df['SURFACE']), p(sorted(df['SURFACE'])),
        'r--', linewidth=2, label='Tendance lineaire')
ax.set_xlabel('Surface (m2)', fontsize=12)
ax.set_ylabel('Prix', fontsize=12)
ax.set_title('Relation entre la Surface et le Prix (couleur = nombre de chambres)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()
print(f"Correlation Surface / Prix : {df['SURFACE'].corr(df['PRIX']):.4f}")

In [ ]:
corr = df.corr()
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black', fontsize=9)
fig.colorbar(im)
ax.set_title('Matrice de correlation', fontsize=14)
plt.tight_layout()
plt.show()
print("\nCorrelation de chaque feature avec PRIX (triee) :")
print(corr['PRIX'].drop('PRIX').sort_values(ascending=False).to_string())

In [ ]:
cat_cols = ['ROUTE_PRINCIPALE', 'CHAMBRE_AMIS', 'SOUS_SOL',
            'CHAUFFAGE_EAU_CHAUDE', 'CLIMATISATION', 'ZONE_PRIVILEGIEE']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Prix moyen selon les equipements (0 = Non, 1 = Oui)', fontsize=14)
for i, col_name in enumerate(cat_cols):
    ax = axes[i // 3, i % 3]
    means = df.groupby(col_name)['PRIX'].mean()
    bars = ax.bar([str(int(k)) for k in means.index], means.values,
                  color=['#e74c3c', '#2ecc71'], edgecolor='black', alpha=0.8)
    ax.set_title(col_name)
    ax.set_ylabel('Prix moyen')
    ax.set_xlabel('0 = Non  /  1 = Oui')
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
furnish_map = {0: 'unfurnished', 1: 'semi-furnished', 2: 'furnished'}
means = df.groupby('STATUT_AMEUBLEMENT_ENC')['PRIX'].mean()
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([furnish_map[k] for k in means.index], means.values,
              color=['#e74c3c', '#3498db', '#2ecc71'], edgecolor='black', alpha=0.8)
for bar, val in zip(bars, means.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:,.0f}', ha='center', va='bottom', fontsize=10)
ax.set_title("Prix moyen par statut d'ameublement", fontsize=13)
ax.set_ylabel('Prix moyen')
plt.tight_layout()
plt.show()

In [ ]:
variable_cible = 'PRIX'

variables_explicatives = {
    'Numeriques'   : ['SURFACE', 'CHAMBRES', 'SALLES_DE_BAIN', 'ETAGES', 'PARKING'],
    'Binaires'     : ['ROUTE_PRINCIPALE', 'CHAMBRE_AMIS', 'SOUS_SOL',
                      'CHAUFFAGE_EAU_CHAUDE', 'CLIMATISATION', 'ZONE_PRIVILEGIEE'],
    'Categorielle' : ['STATUT_AMEUBLEMENT_ENC']
}

print(f"Variable cible : {variable_cible}")
print(f"\nVariables explicatives ({sum(len(v) for v in variables_explicatives.values())} au total) :")
for type_var, cols_list in variables_explicatives.items():
    print(f"\n  {type_var} :")
    for c in cols_list:
        correlation = df[c].corr(df[variable_cible])
        print(f"    - {c:<25} correlation avec PRIX : {correlation:+.4f}")

---
## 6. Preparation des features ML

- Separation features (X) / variable cible (y)
- **Split train/test** avec `random_state=42` pour la reproductibilite

> La normalisation est integree dans le Pipeline sklearn (etape 9).
> On travaille avec des donnees **non scalees** — le Pipeline s'en charge.

In [ ]:
feature_cols = [
    'SURFACE', 'CHAMBRES', 'SALLES_DE_BAIN', 'ETAGES',
    'ROUTE_PRINCIPALE', 'CHAMBRE_AMIS', 'SOUS_SOL',
    'CHAUFFAGE_EAU_CHAUDE', 'CLIMATISATION', 'PARKING',
    'ZONE_PRIVILEGIEE', 'STATUT_AMEUBLEMENT_ENC'
]

X = df[feature_cols]
y = df['PRIX']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train : {X_train.shape[0]} echantillons")
print(f"Test  : {X_test.shape[0]} echantillons")
print(f"\nVariable cible : PRIX")
print(f"Features       : {feature_cols}")

---
## 7. Entrainement des modeles

### Choix des modeles

On compare 3 familles d'algorithmes de regression :

- **Linear Regression** : modele lineaire simple. Sert de reference minimale. Si les autres modeles ne font pas mieux, il y a un probleme.
- **Random Forest** : ensemble d'arbres de decision independants. Robuste, gere bien les relations non-lineaires et les valeurs aberrantes.
- **Gradient Boosting** : arbres construits sequentiellement, chacun corrigeant les erreurs du precedent. Tres performant sur des donnees tabulaires.

Chaque modele est encapsule dans un **Pipeline** avec `StandardScaler` — un seul scaler, coherent du training a l'inference.

In [ ]:
# Chaque modele est un Pipeline : StandardScaler + modele
# On travaille avec X_train et X_test NON scales
pipelines = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LinearRegression())
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  RandomForestRegressor(n_estimators=100, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  GradientBoostingRegressor(n_estimators=100, random_state=42))
    ])
}

trained_pipelines = {}
results = {}
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    trained_pipelines[name] = (pipeline, y_pred)
    results[name] = {
        'MAE':  mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2':   r2_score(y_test, y_pred)
    }
    print(f"\n--- {name} ---")
    print(f"  MAE  : {results[name]['MAE']:,.0f}")
    print(f"  RMSE : {results[name]['RMSE']:,.0f}")
    print(f"  R2   : {results[name]['R2']:.4f}")

---
## 8. Evaluation et comparaison des modeles

**Metriques de regression** : MAE, RMSE, R2

**Metriques de classification** : Accuracy, Precision, Recall
Les prix sont discretises en 3 classes :
- **Classe 0** : prix bas (< 33e percentile)
- **Classe 1** : prix moyen (entre 33e et 66e percentile)
- **Classe 2** : prix eleve (> 66e percentile)

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
print(results_df.to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['MAE',                    'RMSE',                    'R2']
titles  = ['MAE (plus bas = mieux)', 'RMSE (plus bas = mieux)', 'R2 (plus haut = mieux)']
colors  = ['#e74c3c',                '#3498db',                 '#2ecc71']
for i, (metric, title, color) in enumerate(zip(metrics, titles, colors)):
    bars = axes[i].barh(results_df.index, results_df[metric],
                        color=color, edgecolor='black', alpha=0.8)
    axes[i].set_title(title, fontsize=12)
    for bar, val in zip(bars, results_df[metric]):
        label = f'  {val:,.0f}' if metric != 'R2' else f'  {val:.4f}'
        axes[i].text(bar.get_width(), bar.get_y() + bar.get_height()/2,
                     label, va='center', fontsize=10)
plt.tight_layout()
plt.show()
best_model_name = results_df.index[0]
print(f"\nMeilleur modele : {best_model_name} (R2 = {results_df.loc[best_model_name, 'R2']:.4f})")

In [ ]:
# Seuils calcules sur y_train uniquement (pas de data leakage)
p33 = np.percentile(y_train, 33)
p66 = np.percentile(y_train, 66)

def prix_to_classe(prix_series, p33, p66):
    return pd.cut(
        prix_series,
        bins=[-np.inf, p33, p66, np.inf],
        labels=[0, 1, 2]
    ).astype(int)

y_test_classes = prix_to_classe(y_test, p33, p66)

print(f"Seuils : Classe 0 < {p33:,.0f} | {p33:,.0f} <= Classe 1 < {p66:,.0f} | Classe 2 >= {p66:,.0f}")

classif_results = {}
for name, (pipeline, y_pred) in trained_pipelines.items():
    y_pred_classes = prix_to_classe(pd.Series(y_pred), p33, p66)
    classif_results[name] = {
        'Accuracy':  accuracy_score(y_test_classes, y_pred_classes),
        'Precision': precision_score(y_test_classes, y_pred_classes, average='macro', zero_division=0),
        'Recall':    recall_score(y_test_classes, y_pred_classes, average='macro', zero_division=0)
    }

classif_df = pd.DataFrame(classif_results).T.sort_values('Accuracy', ascending=False)
print("\n=== Metriques de classification ===")
print(classif_df.round(4).to_string())

In [ ]:
best_pipeline, best_y_pred = trained_pipelines[best_model_name]
y_pred_classes_best = prix_to_classe(pd.Series(best_y_pred), p33, p66)
cm = confusion_matrix(y_test_classes, y_pred_classes_best)
labels = ['Bas', 'Moyen', 'Eleve']
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel('Classe predite', fontsize=12)
ax.set_ylabel('Classe reelle', fontsize=12)
ax.set_title(f'Matrice de confusion — {best_model_name}', fontsize=13)
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
fig.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Metriques de classification par modele', fontsize=14)
metrics_c = ['Accuracy', 'Precision', 'Recall']
colors_c  = ['#9b59b6',  '#e67e22',   '#1abc9c']
for i, (metric, color) in enumerate(zip(metrics_c, colors_c)):
    bars = axes[i].barh(classif_df.index, classif_df[metric],
                        color=color, edgecolor='black', alpha=0.8)
    axes[i].set_title(f'{metric} (plus haut = mieux)', fontsize=12)
    axes[i].set_xlim(0, 1)
    for bar, val in zip(bars, classif_df[metric]):
        axes[i].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                     f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

---
## 9. Optimisation avec Grid Search

### Hyperparametres optimises et leur role

- **n_estimators** : nombre d'arbres construits. Plus il est eleve, plus le modele est precis mais plus lent a entrainer.
- **max_depth** : profondeur maximale de chaque arbre. Une valeur trop elevee provoque du surapprentissage (overfitting).
- **min_samples_split** : nombre minimum d'echantillons pour diviser un noeud. Une valeur elevee simplifie le modele.
- **min_samples_leaf** : nombre minimum d'echantillons dans une feuille. Evite les feuilles trop specifiques.

### Methode
Grid Search avec validation croisee **5-fold** sur le **Pipeline complet** (StandardScaler + GradientBoosting).
Les hyperparametres du modele sont prefixes par `model__` pour cibler l'etape correcte du Pipeline.

In [ ]:
# Grid Search sur le Pipeline complet
# Les hyperparametres sont prefixes par 'model__' pour cibler l'etape 'model' du Pipeline
pipeline_gb = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  GradientBoostingRegressor(random_state=42))
])

param_grid = {
    'model__n_estimators':      [100, 200, 300],
    'model__max_depth':         [3, 5, 7],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf':  [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator  = pipeline_gb,
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'r2',
    n_jobs     = -1,
    verbose    = 1
)

# On passe X_train NON scale — le Pipeline gere le scaling
grid_search.fit(X_train, y_train)

best_pipeline_opt = grid_search.best_estimator_
y_pred_opt        = best_pipeline_opt.predict(X_test)

print(f"Meilleurs hyperparametres : {grid_search.best_params_}")
print(f"Meilleur R2 (CV 5-fold)   : {grid_search.best_score_:.4f}")
print(f"\n--- Gradient Boosting optimise ---")
print(f"  MAE  : {mean_absolute_error(y_test, y_pred_opt):,.0f}")
print(f"  RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_opt)):,.0f}")
print(f"  R2   : {r2_score(y_test, y_pred_opt):.4f}")

In [ ]:
# AVANT : Gradient Boosting de base (Pipeline etape 7)
_, y_pred_base = trained_pipelines['Gradient Boosting']
y_pred_base_classes = prix_to_classe(pd.Series(y_pred_base), p33, p66)

# APRES : Pipeline optimise par Grid Search
y_pred_opt_classes = prix_to_classe(pd.Series(y_pred_opt), p33, p66)

avant = {
    'MAE':       mean_absolute_error(y_test, y_pred_base),
    'RMSE':      np.sqrt(mean_squared_error(y_test, y_pred_base)),
    'R2':        r2_score(y_test, y_pred_base),
    'Accuracy':  accuracy_score(y_test_classes, y_pred_base_classes),
    'Precision': precision_score(y_test_classes, y_pred_base_classes, average='macro', zero_division=0),
    'Recall':    recall_score(y_test_classes, y_pred_base_classes, average='macro', zero_division=0)
}
apres = {
    'MAE':       mean_absolute_error(y_test, y_pred_opt),
    'RMSE':      np.sqrt(mean_squared_error(y_test, y_pred_opt)),
    'R2':        r2_score(y_test, y_pred_opt),
    'Accuracy':  accuracy_score(y_test_classes, y_pred_opt_classes),
    'Precision': precision_score(y_test_classes, y_pred_opt_classes, average='macro', zero_division=0),
    'Recall':    recall_score(y_test_classes, y_pred_opt_classes, average='macro', zero_division=0)
}

print(f"{'Metrique':<12} {'Avant':>12} {'Apres':>12} {'Gain':>12}")
print("-" * 50)
for metric in ['MAE', 'RMSE', 'R2', 'Accuracy', 'Precision', 'Recall']:
    a, b = avant[metric], apres[metric]
    gain = b - a
    if metric in ['MAE', 'RMSE']:
        print(f"{metric:<12} {a:>12,.0f} {b:>12,.0f} {gain:>+12,.0f}")
    else:
        print(f"{metric:<12} {a:>12.4f} {b:>12.4f} {gain:>+12.4f}")

In [ ]:
# Extraction du modele depuis le Pipeline
best_gb_model = best_pipeline_opt.named_steps['model']
feat_imp = pd.Series(best_gb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.plot(kind='barh', color='steelblue', edgecolor='black', alpha=0.8, ax=ax)
ax.set_title('Importance des features - Gradient Boosting optimise', fontsize=14)
ax.set_xlabel('Importance relative')
plt.tight_layout()
plt.show()
print("Feature la plus importante :", feat_imp.idxmax())

---
## 10. Enregistrement dans le Snowflake Model Registry

Deux versions enregistrees :
- `v1` : Pipeline Gradient Boosting de base
- `v2` : Pipeline Gradient Boosting optimise — candidat **production**

Le Pipeline est enregistre tel quel — scaler inclus.

In [ ]:
reg = Registry(session=session, database_name='HOUSE_PRICE_DB', schema_name='ML')

# v1 : Pipeline Gradient Boosting de base (deja entraine a l'etape 7)
pipeline_v1, y_pred_v1 = trained_pipelines['Gradient Boosting']
v1_r2  = r2_score(y_test, y_pred_v1)

y_pred_v1_classes = prix_to_classe(pd.Series(y_pred_v1), p33, p66)
v1_acc = accuracy_score(y_test_classes, y_pred_v1_classes)
v1_pre = precision_score(y_test_classes, y_pred_v1_classes, average='macro', zero_division=0)
v1_rec = recall_score(y_test_classes, y_pred_v1_classes, average='macro', zero_division=0)

try:
    reg.delete_model('house_price_predictor')
except Exception:
    pass

reg.log_model(
    model             = pipeline_v1,
    model_name        = 'house_price_predictor',
    version_name      = 'v1',
    sample_input_data = X_test.head(10),
    target_platforms  = [TargetPlatform.WAREHOUSE],
    metrics = {
        'MAE':       float(mean_absolute_error(y_test, y_pred_v1)),
        'RMSE':      float(np.sqrt(mean_squared_error(y_test, y_pred_v1))),
        'R2':        float(v1_r2),
        'Accuracy':  float(v1_acc),
        'Precision': float(v1_pre),
        'Recall':    float(v1_rec)
    },
    comment = f'Pipeline GradientBoosting de base — R2={v1_r2:.4f} Accuracy={v1_acc:.4f}'
)
print(f"v1 enregistree — R2={v1_r2:.4f} | Accuracy={v1_acc:.4f}")

In [ ]:
# v2 : Pipeline optimise par Grid Search (deja entraine a l'etape 9)
v2_r2   = r2_score(y_test, y_pred_opt)
v2_mae  = mean_absolute_error(y_test, y_pred_opt)
v2_rmse = np.sqrt(mean_squared_error(y_test, y_pred_opt))

v2_acc = accuracy_score(y_test_classes, y_pred_opt_classes)
v2_pre = precision_score(y_test_classes, y_pred_opt_classes, average='macro', zero_division=0)
v2_rec = recall_score(y_test_classes, y_pred_opt_classes, average='macro', zero_division=0)

reg.log_model(
    model             = best_pipeline_opt,
    model_name        = 'house_price_predictor',
    version_name      = 'v2',
    sample_input_data = X_test.head(10),
    target_platforms  = [TargetPlatform.WAREHOUSE],
    metrics = {
        'MAE':       float(v2_mae),
        'RMSE':      float(v2_rmse),
        'R2':        float(v2_r2),
        'Accuracy':  float(v2_acc),
        'Precision': float(v2_pre),
        'Recall':    float(v2_rec)
    },
    comment = f'Pipeline optimise — PRODUCTION — R2={v2_r2:.4f} Accuracy={v2_acc:.4f}'
)
print(f"v2 enregistree — R2={v2_r2:.4f} | Accuracy={v2_acc:.4f}")
print(f"\nComparaison v1 vs v2 :")
print(f"  R2        : v1={v1_r2:.4f}  ->  v2={v2_r2:.4f}")
print(f"  Accuracy  : v1={v1_acc:.4f}  ->  v2={v2_acc:.4f}")
print(f"  Precision : v1={v1_pre:.4f}  ->  v2={v2_pre:.4f}")
print(f"  Recall    : v1={v1_rec:.4f}  ->  v2={v2_rec:.4f}")
print("\n-> v2 selectionnee pour la production.")

In [ ]:
print("=== Modeles dans le Registry ===")
for m in reg.models():
    print(f"\nModele : {m.name}")
    for v in m.versions():
        print(f"  Version     : {v.version_name}")
        for metric_name in ['MAE', 'RMSE', 'R2', 'Accuracy', 'Precision', 'Recall']:
            try:
                print(f"    {metric_name} : {v.get_metric(metric_name)}")
            except Exception:
                pass
        print(f"  Commentaire : {v.comment}")
print("\n-> v2 est le modele retenu pour la production.")

---
## 11. Inference avec le modele enregistre

Trois scenarios :
1. Inference sur des donnees de test (comparaison avec valeurs reelles)
2. Inference sur de nouvelles donnees simulees (scenario metier)
3. Inference sur toute la table Gold avec sauvegarde dans Snowflake

In [ ]:
mv_prod = reg.get_model('house_price_predictor').version('v2')

# Donnees NON scalees — le Pipeline gere le scaling
test_sample = X_test.head(20).copy()
predictions = mv_prod.run(test_sample, function_name='predict')

result_df = test_sample.copy()
result_df['PRIX_REEL']   = y_test.head(20).values
result_df['PRIX_PREDIT'] = predictions['output_feature_0'].values
result_df['ERREUR']      = abs(result_df['PRIX_REEL'] - result_df['PRIX_PREDIT'])
result_df['ERREUR_PCT']  = (result_df['ERREUR'] / result_df['PRIX_REEL'] * 100).round(1)

print(result_df[['PRIX_REEL', 'PRIX_PREDIT', 'ERREUR', 'ERREUR_PCT']].to_string())
print(f"\nErreur moyenne : {result_df['ERREUR_PCT'].mean():.1f}%")

In [ ]:
# 3 nouvelles maisons jamais vues par le modele
# Donnees NON scalees — le Pipeline gere le scaling
nouvelles_maisons = pd.DataFrame([
    {
        'SURFACE': 300, 'CHAMBRES': 5, 'SALLES_DE_BAIN': 3, 'ETAGES': 2,
        'ROUTE_PRINCIPALE': 1, 'CHAMBRE_AMIS': 1, 'SOUS_SOL': 1,
        'CHAUFFAGE_EAU_CHAUDE': 1, 'CLIMATISATION': 1, 'PARKING': 3,
        'ZONE_PRIVILEGIEE': 1, 'STATUT_AMEUBLEMENT_ENC': 2
    },
    {
        'SURFACE': 80, 'CHAMBRES': 2, 'SALLES_DE_BAIN': 1, 'ETAGES': 1,
        'ROUTE_PRINCIPALE': 1, 'CHAMBRE_AMIS': 0, 'SOUS_SOL': 0,
        'CHAUFFAGE_EAU_CHAUDE': 0, 'CLIMATISATION': 0, 'PARKING': 1,
        'ZONE_PRIVILEGIEE': 0, 'STATUT_AMEUBLEMENT_ENC': 0
    },
    {
        'SURFACE': 150, 'CHAMBRES': 3, 'SALLES_DE_BAIN': 2, 'ETAGES': 1,
        'ROUTE_PRINCIPALE': 1, 'CHAMBRE_AMIS': 0, 'SOUS_SOL': 1,
        'CHAUFFAGE_EAU_CHAUDE': 0, 'CLIMATISATION': 1, 'PARKING': 2,
        'ZONE_PRIVILEGIEE': 0, 'STATUT_AMEUBLEMENT_ENC': 1
    }
], columns=feature_cols)

predictions_new = mv_prod.run(nouvelles_maisons, function_name='predict')
nouvelles_maisons['PRIX_PREDIT'] = predictions_new['output_feature_0'].values

descriptions = [
    'Maison 1 : grande maison meublee, tous equipements, zone privilegiee',
    'Maison 2 : petit appartement non meuble',
    'Maison 3 : maison intermediaire semi-meublee'
]
classe_labels = {0: 'Bas', 1: 'Moyen', 2: 'Eleve'}

print("=== Predictions sur nouvelles maisons ===")
for desc, prix in zip(descriptions, nouvelles_maisons['PRIX_PREDIT']):
    classe = prix_to_classe(pd.Series([prix]), p33, p66).values[0]
    print(f"\n{desc}")
    print(f"  Prix estime   : {prix:,.0f}")
    print(f"  Gamme de prix : {classe_labels[classe]}")
print(f"\nPrix moyen du dataset : {df['PRIX'].mean():,.0f}")

In [ ]:
full_df         = session.table('HOUSE_PRICE_DB.GOLD.HOUSE_PRICES').to_pandas()
full_features   = full_df[feature_cols]
all_predictions = mv_prod.run(full_features, function_name='predict')

result_table = full_df.copy()
result_table['PRIX_PREDIT'] = all_predictions['output_feature_0'].values

session.create_dataframe(result_table).write.mode('overwrite').save_as_table(
    'HOUSE_PRICE_DB.ML.HOUSE_PRICES_PREDICTIONS'
)
print(f"Predictions sauvegardees dans HOUSE_PRICE_DB.ML.HOUSE_PRICES_PREDICTIONS")
print(f"{len(result_table)} lignes avec predictions")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(result_table['PRIX'], result_table['PRIX_PREDIT'],
           alpha=0.5, color='steelblue', edgecolor='black', s=30)
min_val = min(result_table['PRIX'].min(), result_table['PRIX_PREDIT'].min())
max_val = max(result_table['PRIX'].max(), result_table['PRIX_PREDIT'].max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Prediction parfaite')
ax.set_xlabel('Prix reel',   fontsize=12)
ax.set_ylabel('Prix predit', fontsize=12)
ax.set_title('Prix reel vs Prix predit — Pipeline Gradient Boosting optimise (v2)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()
print(f"R2 final : {r2_score(result_table['PRIX'], result_table['PRIX_PREDIT']):.4f}")